# Sprawozdanie z laboratorium nr 5

## Agent SAC z buforem HER w środowisku PandaReach-v3


### Zapis wag i metadanych

W ramach implementacji wprowadzono trwały zapis wyniku treningu: wagi polityki zapisuje plik `policy.pt`, natomiast konfiguracja eksperymentu — dokument `metadata.json`. Umożliwia to powtórne użycie nauczonej polityki bez ponownego uruchamiania pełnego treningu.

Wagi same w sobie nie definiują jednoznacznie modelu: muszą odpowiadać architekturze sieci (m.in. rozmiary warstw, typ ekstraktora cech z obserwacji), parametrom algorytmu SAC oraz ustawieniom bufora HER. W pliku `metadata.json` pole `signature` gromadzi te informacje: identyfikator środowiska, opis polityki, konfigurację HER, argumenty konstruktora `SAC`, liczbę kroków treningu i parametry ewaluacji. Przy wczytywaniu sygnatura odczytana z dysku jest porównywana z sygnaturą wyliczoną w kodzie funkcją `canonical_signature()` w module `train.py`; niespójność powoduje przerwanie działania programu, co zapobiega błędnemu dopasowaniu wag do innej konfiguracji.

Dodatkowo zapisywane są m.in. znacznik czasu `saved_at_utc` oraz, w razie podania przy zapisie, ścieżka `tensorboard_log_dir` do logów TensorBoard.

**Rysunek 1.** Zrzut ekranu przedstawiający zapis runu treningowego: struktura katalogu, fragment `metadata.json` z polem `signature` oraz plik wag `policy.pt`.

![Rysunek 1 — zapis runu (metadata i wagi)](<Screenshot 2026-05-30 202306.png>)

### Wczytywanie runu i interfejs wiersza poleceń

Po zakończeniu treningu w katalogu runu (np. `weights/2026-05-24_16-53/`) znajdują się `metadata.json` oraz `policy.pt`. Wczytanie odbywa się funkcją `load_sac_from_run_dir`: odczyt metadanych, weryfikacja sygnatury (`_assert_signature_matches`), utworzenie środowiska i obiektu `SAC` przez `instantiate_sac_from_signature`, następnie `algo.load(...)`. Tryb wyłącznie ewaluacji (render bez treningu) uruchamia się poleceniem `python train.py --load <ścieżka_do_katalogu_runu>`.

Poniżej zamieszczono wybrane fragmenty modułu `train.py` ilustrujące powyższy mechanizm.


```python
# train.py — sygnatura eksperymentu (zgodność zapisu z definicją w kodzie)
def canonical_signature(sac_branch: dict[str, Any] | None = None) -> dict[str, Any]:
    sac = dict(SAC_KWARGS) if sac_branch is None else dict(sac_branch)
    return {
        "format_version": 1,
        "env_id": DEFAULT_ENV_ID,
        "policy": {
            "hidden_sizes": list(POLICY_HIDDEN),
            "extractor": POLICY_EXTRACTOR,
        },
        "her": {
            "n_sampled_goal": HER_N_SAMPLED_GOAL,
            "goal_selection_strategy": HER_STRATEGY,
            "replay_buffer_size_train": REPLAY_BUFFER_SIZE_TRAIN,
            "replay_buffer_size_load": REPLAY_BUFFER_SIZE_LOAD,
        },
        "sac": sac,
        "train_loop": {"n_steps": TRAIN_N_STEPS, "log_interval": TRAIN_LOG_INTERVAL},
        "eval_render": {"n_episodes": EVAL_N_EPISODES, "sleep": EVAL_RENDER_SLEEP},
    }

def save_training_run(algo: SAC, run_dir: Path, *, sac_kwargs=None, tensorboard_log_dir=None) -> None:
    run_dir.mkdir(parents=True, exist_ok=True)
    effective_sac = dict(SAC_KWARGS) if sac_kwargs is None else dict(sac_kwargs)
    sig = canonical_signature(sac_branch=effective_sac)
    doc = {
        "signature": sig,
        "weights_file": WEIGHTS_FILENAME,
        "saved_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    tb_meta = _tensorboard_dir_for_metadata(tensorboard_log_dir)
    if tb_meta is not None:
        doc["tensorboard_log_dir"] = tb_meta
    meta_path = run_dir / METADATA_FILENAME
    with meta_path.open("w", encoding="utf-8") as f:
        json.dump(doc, f, indent=2, sort_keys=True)
        f.write("\n")
    algo.save(str(run_dir / WEIGHTS_FILENAME))

def load_sac_from_run_dir(run_dir: str | Path) -> Tuple[gym.Env, SAC, dict[str, Any]]:
    run_path = Path(run_dir).expanduser().resolve()
    meta_path = run_path / METADATA_FILENAME
    with meta_path.open(encoding="utf-8") as f:
        doc = json.load(f)
    _assert_signature_matches(doc["signature"])
    weights_path = run_path / doc.get("weights_file", WEIGHTS_FILENAME)
    device = resolve_device()
    env, algo = instantiate_sac_from_signature(doc["signature"], device=device)
    algo.load(str(weights_path))
    return env, algo, doc

# train.py — fragment definicji argumentów CLI
parser.add_argument(
    "--load", type=Path, metavar="RUN_DIR",
    help="Katalog runu (metadata.json + policy.pt)",
)
args = parser.parse_args()
if args.load is None:
    main_train()
else:
    run_eval_render_from_run_dir(args.load.expanduser().resolve())
```


### Automatyczna temperatura entropii w algorytmie SAC

W implementacji zastosowano automatyczne dostrajanie współczynnika entropii $\alpha$ zgodnie z opisem w pracy Haarnoja et al. (*Soft Actor-Critic Algorithms and Applications*, [arXiv:1812.05905](https://arxiv.org/abs/1812.05905)). Zamiast stałej wartości $\alpha$ optymalizowany jest parametr $\log\alpha$, przy czym $\alpha = \exp(\log\alpha)$ pełni rolę temperatury w wyrażeniu straty maksymalnej entropii.

Przy ustawieniu `target_entropy="auto"` przyjmuje się docelową entropię polityki równą $-\dim(\mathcal{A})$, tj. ujemnej liczbie wymiarów przestrzeni akcji (iloczyn wymiarów wektora akcji), co jest standardowym wyborem w implementacjach SAC dla przestrzeni ciągłych.

Współczynnik $\alpha$ wchodzi w skład kopii Bellmana dla funkcji $Q$ oraz w strate polityki. Aktualizacja $\log\alpha$ opiera się na wartościach $\log\pi$ z kroku aktualizacji polityki; w celu uniknięcia niepożądanego przepływu gradientu współczynnik $\alpha$ (lub $\exp(\log\alpha)$) w stratach $Q$ i $\pi$ jest odłączany od grafu obliczeniowego względem optymalizacji $\alpha$, zgodnie z typową praktyką implementacyjną opartą na dyskusjach przy repozytorium Softlearning ([issue #60](https://github.com/rail-berkeley/softlearning/issues/60)).

W module `train.py` w słowniku `SAC_KWARGS` ustawiono `alpha="auto"` oraz `target_entropy="auto"`. Poniżej zestawiono kluczowe fragmenty implementacji z pliku `asdf/algos.py`: inicjalizacja parametru $\log\alpha$, obliczanie strat, krok optymalizatora oraz zapis i odczyt stanu uczonej temperatury w checkpointcie.


```python
# asdf/algos.py — fragment konstruktora SAC: tryb alpha="auto"
if isinstance(alpha, str) and alpha.startswith("auto"):
    init_value = 1.0
    if "_" in alpha:
        init_value = float(alpha.split("_")[1])
    self.log_alpha = nn.Parameter(
        torch.tensor(np.log(init_value), dtype=torch.float32, device=alpha_device)
    )
    self.alpha_optimizer = Adam([self.log_alpha], lr=lr)
    with torch.no_grad():
        self.alpha = self.log_alpha.exp().detach()
else:
    self.alpha = torch.tensor(float(alpha), dtype=torch.float32, device=alpha_device)
    self.alpha_optimizer = None
    self.log_alpha = None

def _entropy_coefficient(self, detach: bool) -> torch.Tensor:
    if self.alpha_optimizer is None:
        ent = self.alpha
    else:
        ent = self.log_alpha.exp()
    return ent.detach() if detach else ent

# Kopie Bellmana dla Q: współczynnik entropii bez gradientu względem alpha
ent_coef = self._entropy_coefficient(detach=True)
backup = r + self.gamma * (1 - ter) * (q_pi_targ - ent_coef * logp_a2)

# Strata polityki
loss_pi = (self._entropy_coefficient(detach=True) * logp_pi - q_pi).mean()

def compute_loss_alpha(self, logp_pi):
    if self.alpha_optimizer is None:
        return torch.tensor(0.0, device=logp_pi.device, dtype=logp_pi.dtype), self.alpha
    alpha_loss = -(self.log_alpha * (logp_pi + self.target_entropy).detach()).mean()
    return alpha_loss, self.log_alpha.exp()

# Aktualizacja log_alpha po aktualizacji pi i sieci docelowych
if self.alpha_optimizer is not None:
    alpha_loss, _ = self.compute_loss_alpha(logp_pi)
    self.alpha_optimizer.zero_grad()
    alpha_loss.backward()
    self.alpha_optimizer.step()
    with torch.no_grad():
        self.alpha = self.log_alpha.exp().detach()

# Zapis i odczyt: osobna obsługa uczonych alpha / log_alpha
def save(self, path: str):
    payload = {
        "policy_state_dict": self.policy.state_dict(),
        "pi_optimizer_state_dict": self.pi_optimizer.state_dict(),
        "q_optimizer_state_dict": self.q_optimizer.state_dict(),
    }
    if self.alpha_optimizer is not None:
        payload["log_alpha"] = self.log_alpha.detach().cpu()
        payload["alpha_optimizer_state_dict"] = self.alpha_optimizer.state_dict()
    else:
        payload["alpha"] = self.alpha
    torch.save(payload, path)

def load(self, path: str):
    checkpoint = torch.load(path, map_location=next(self.policy.parameters()).device)
    dev = next(self.policy.parameters()).device
    # ... load_state_dict: policy, pi_optimizer, q_optimizer ...
    if "log_alpha" in checkpoint and self.alpha_optimizer is not None:
        self.log_alpha.data.copy_(checkpoint["log_alpha"].to(dtype=self.log_alpha.dtype, device=dev))
        self.alpha_optimizer.load_state_dict(checkpoint["alpha_optimizer_state_dict"])
        with torch.no_grad():
            self.alpha = self.log_alpha.exp().detach()
    else:
        self.alpha = checkpoint["alpha"].to(device=dev)
```


### Hindsight Experience Replay (`HerReplayBuffer`)

Hindsight Experience Replay (HER) polega na ponownym zapisie przejść z epizodu z alternatywnym polem `desired_goal` w obserwacji, tak aby nagroda była liczona względem nowego celu. W implementacji przejścia gromadzone są w buforze epizodycznym do momentu wywołania `end_episode`; następnie każde przejście zapisywane jest raz z oryginalnym celem oraz `n_sampled_goal` razy z celem wylosowanym według strategii. Zastosowano strategię `future`: jako kandydaci na nowy cel występują stany `achieved_goal` z przyszłych kroków tego samego epizodu; nagroda wyliczana jest funkcją `compute_reward` środowiska (`env.unwrapped`). W pliku `train.py` przyjęto `HER_STRATEGY = "future"` oraz `HER_N_SAMPLED_GOAL = 3`.

Fragment implementacji z modułu `asdf/buffers.py` zamieszczono poniżej.


```python
# asdf/buffers.py — zakończenie epizodu: zapis oryginalny + warianty HER
# (w pliku źródłowym: from copy import deepcopy)
def end_episode(self) -> None:
    ep = self._episode_transitions
    if not ep:
        return
    env_compute = self.env.unwrapped
    for t_idx, tr in enumerate(ep):
        self._store_one_transition(
            tr["observation"], tr["action"], tr["reward"],
            tr["next_observation"], tr["terminated"], tr["truncated"], tr["info"],
        )
        for _ in range(self.n_sampled_goal):
            new_goal = self._sample_goal(ep, t_idx)
            o = self._copy_obs(tr["observation"])
            o2 = self._copy_obs(tr["next_observation"])
            o["desired_goal"] = np.asarray(new_goal, dtype=np.float32).copy()
            o2["desired_goal"] = np.asarray(new_goal, dtype=np.float32).copy()
            info = deepcopy(tr["info"])
            r = float(
                env_compute.compute_reward(
                    o2["achieved_goal"], o2["desired_goal"], info
                )
            )
            self._store_one_transition(
                o, tr["action"], r, o2, tr["terminated"], tr["truncated"], info,
            )

def _sample_goal(self, episode, transition_idx: int) -> NDArray:
    if self.selection_strategy == "future":
        candidates = [
            np.asarray(episode[j]["next_observation"]["achieved_goal"], dtype=np.float32).copy()
            for j in range(transition_idx + 1, len(episode))
        ]
        if not candidates:
            return self._sample_goal_episode(episode)
        pick = int(self._rng.integers(0, len(candidates)))
        return np.asarray(candidates[pick], dtype=np.float32).copy()
    # strategie "final" oraz "episode" — jak w pliku źródłowym
```


### Wyniki: krzywe uczące z TensorBoard

W trakcie treningu rejestrowano skalary w module `asdf/algos.py`: straty `loss_q`, `loss_pi`, `loss_alpha`, wartość `alpha`, a także `ep_return`, `ep_length` oraz metryki okresowej ewaluacji deterministycznej `test_ep_return`, `test_ep_length`. Na osi odciętych wykresów występuje zwykle indeks kroku środowiska $t$. Poniżej zestawiono załączone rysunki w kolejności prezentacji; szczegółowe wartości liczbowe odczytuje się bezpośrednio z osi na plikach graficznych.

**Rysunek 2.** Zestawienie metryk w TensorBoard (`first.png`).

![Rysunek 2](first.png)

Widok zbiorczy ułatwia jednoczesną ocenę dynamiki kilku sygnałów treningowych i testowych.

**Rysunek 3.** Przebieg współczynnika temperatury entropii $\alpha$ (`alpha.png`).

![Rysunek 3](<alpha.png>)

Krzywa ilustruje adaptację poziomu eksploracji: przy uczeniu $\log\alpha$ wartość $\alpha$ może stabilizować się lub zmieniać się w odpowiedzi na odchylenie entropii polityki od wartości docelowej.

**Rysunek 4.** Strata związana z uaktualnianiem $\log\alpha$ (`loss_alpha.png`).

![Rysunek 4](<loss_alpha.png>)

Wielkość `loss_alpha` odpowiada funkcji celu dla parametru $\log\alpha$. Zbliżanie się krzywej do małych wartości świadczy o dopasowaniu poziomu entropii do przyjętego `target_entropy`.

**Rysunek 5.** Strata polityki (`loss_pi.png`).

![Rysunek 5](<loss_pi.png>)

Strata aktora w SAC ma postać średniej z wyrażenia $\alpha \log\pi(a|s) - Q(s,a)$ dla stochastycznej polityki użytej w kroku gradientu. Przebieg nie musi być monotoniczny ze względu na równoległą aktualizację sieci $Q$ oraz $\alpha$.

**Rysunek 6.** Strata funkcji wartości (`loss_q.png`).

![Rysunek 6](<loss_q.png>)

Wartość `loss_q` jest średnią strat MSE obu krytyków względem celu Bellmana z siecią docelową i członem entropijnym. Na początku treningu obserwuje się zwykle większą zmienność, później krzywa częściej się wygładza.

**Rysunek 7.** Strata funkcji wartości — drugi zrzut (`loss_q_2.png`).

![Rysunek 7](<loss_q_2.png>)

Drugi zapis tej samej metryki może odpowiadać innemu zakresowi osi albo innemu fragmentowi przebiegu; służy do uściślenia odczytu w wybranym przedziale kroków.

**Rysunek 8.** Długość epizodu w fazie zbierania danych (`ep_length.png`).

![Rysunek 8](<ep_length.png>)

Metryka dotyczy epizodów zbieranych z użyciem polityki eksploracyjnej (w tym fazy losowej na początku). Skrócenie epizodu bywa związane z szybszym osiągnięciem celu w zadaniu typu reach.

**Rysunek 9.** Średnia długość epizodu w próbie deterministycznej (`test_ep_length.png`).

![Rysunek 9](<test_ep_length.png>)

Wartość uśredniana jest okresowo po `n_test_episodes` epizodach z deterministycznym wyborem akcji. Spadek średniej długości wskazuje na skracanie ścieżki do celu przy ewaluacji.

**Rysunek 10.** Średni zwrot w próbie deterministycznej (`test_ep_return.png`).

![Rysunek 10](<test_ep_return.png>)

Rosnący (mniej ujemny) zwrot testowy świadczy o poprawie jakości polityki w środowisku PandaReach przy typowej funkcji nagrody.

---

### Wnioski

Zrealizowano trening agenta SAC z buforem HER dla środowiska PandaReach-v3, z automatycznym doborem temperatury entropii oraz z mechanizmem zapisu i weryfikowanego odczytu runu (`metadata.json`, `policy.pt`). Połączenie HER z SAC poprawia wykorzystanie próbek z epizodów, w których pierwotny cel nie został osiągnięty, natomiast adaptacyjne $\alpha$ utrzymuje regulację entropii bez ręcznego strojenia stałego współczynnika